# 06 · Model comparison (Gemini primary vs fallback)

Compares models only on cases each actually answered; no claim of superiority without a controlled comparison (same questions, both models answering).

**Inputs** are versioned files under `research/data/` and `research/configs/`; nothing here connects to the production database or writes to it. Run top to bottom (Kernel → Restart & Run All).

In [ ]:
import sys, json, platform, warnings
from pathlib import Path
import numpy as np, pandas as pd, scipy, matplotlib
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

RESEARCH = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd() / "research"
sys.path.insert(0, str(RESEARCH / "scripts"))
import analysis as A

A.style()
np.random.seed(A.SEED)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
EXPERIMENTS = json.loads((RESEARCH / "configs/experiments.json").read_text())
print(f"python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__} | "
      f"scipy {scipy.__version__} | matplotlib {matplotlib.__version__} | seed {A.SEED}")
print(f"dataset snapshot: {A.SNAPSHOT} (set RAG_SNAPSHOT=corpus_v1|corpus_v2) | results: {A.RESULTS.relative_to(RESEARCH)} | "
      f"figures: {A.FIGURES.relative_to(RESEARCH)}")

In [ ]:
f = A.RESULTS / "answers_production_heldout_v0.2.jsonl"
answers = pd.read_json(f, lines=True) if f.exists() else pd.DataFrame()
ann_f = A.DATA / "review" / "answer_annotation_template.csv"
annotations = pd.read_csv(ann_f) if ann_f.exists() else pd.DataFrame()
print(f"{len(answers)} recorded answer cases" if len(answers) else "Not evaluated: no answer-evaluation run found")

In [ ]:
if len(answers):
    configured = EXPERIMENTS["generation"]
    display(pd.DataFrame([("configured primary", configured["configured_primary_model"]),
                          ("configured fallback", configured["configured_fallback_model"]),
                          ("prompt version", configured["prompt_version"])], columns=["setting", "value"]))
    by_model = answers.assign(model=answers["model"].fillna("(no answer generated)")).groupby("model").agg(
        cases=("question_id", "size"), fallback_used=("fallback_used", "mean"),
        median_wall_ms=("wall_latency_ms", "median"), median_total_tokens=("total_tokens", "median"),
        language_consistent=("language_consistent", "mean"), citation_hit=("citation_hit", "mean"))
    display(by_model.round(3))
    served = set(answers.loc[answers["answer_generation"] == "llm", "model"].dropna())
    primary_ok = any(configured["configured_primary_model"].split(" ")[0] in m for m in served)
    display(Markdown(("" if primary_ok else f"""**Primary model: Not evaluated.** No case in this run was answered by the configured
primary model (served models: {', '.join(sorted(served)) or 'none'}); a primary-vs-fallback comparison of quality,
latency or tokens is therefore not possible. """) + f"""Fallback frequency in this run:
{answers['fallback_used'].mean():.0%} of generated answers. Cost is not reported (no reliable billing data)."""))